# BAIT-Enhanced — Complete Colab Notebook (Group 7)

**TOP_K_FILTER pruning + parallel initial-token scanning**, grafted onto BAIT-Enhanced.

This single notebook contains everything. Run **Setup** once, then jump to any path:

| Section | What it does | Needs |
|---|---|---|
| **Setup** | clone repo + base install | — |
| **Path 1** | black-box / stub scan | nothing (CPU, seconds) |
| **Path 2** | real Ollama model scan | small model download |
| **Path 3** | full GPU model-zoo scan | **A100 GPU**, HF token |
| **Graphs** | result charts | Section A: nothing · Section B: a Path-3 run |

> For a quick demo use **Path 1** + **Graphs → Section A**.
> For the full pipeline use **Path 3** + **Graphs → Section B**.


---
## Setup (run once)

Clones the repo and installs the light dependencies shared by all paths. Safe to re-run.


In [ ]:
%cd /content
![ -d BAIT-Enhanced-group7 ] || git clone https://github.com/sureshbabugandla/BAIT-Enhanced-group7.git
%cd /content/BAIT-Enhanced-group7
!pip install -q numpy tqdm matplotlib pandas
print("\nCurrent dir:"); !pwd
print("Scan scripts:"); !ls scripts/ | grep scan

---
# Path 1 — Black-box / stub scan  (no GPU, instant)

Runs TOP_K_FILTER pruning + parallel initial-token scanning against a built-in stub model
with a *planted* backdoor target. Nothing to download; finishes in seconds. Recommended demo path.


### 1.1 Run the scan
Should recover `"Click http://malicious for more info"` with Q-Score **0.95**, scanning only the
top-50 candidate tokens across 8 parallel workers.

In [ ]:
!python scripts/scan_blackbox.py --backend stub --top-k-filter 50 --parallel-workers 8

### 1.2 Parallel speedup (verdict-preserving)
Same scan, different worker counts. Wall-clock time drops while the **best token stays identical**.

In [ ]:
import time
from src.core.parallel_scan import parallel_initial_token_scan

def score(tid):
    time.sleep(0.01)                 # simulate a blocking model call
    return (tid / 100.0, f"target_{tid}", {})

ids = list(range(60))
print("workers |  time  | best token")
print("--------|--------|-----------")
base = None
for w in [1, 2, 4, 8]:
    t = time.time()
    r = parallel_initial_token_scan(ids, score, max_workers=w, show_progress=False)
    dt = time.time() - t
    if base is None: base = dt
    print(f"   {w:<4} | {dt:5.2f}s | {r.best.token_id}   (speedup {base/dt:.1f}x)")

### 1.3 TOP_K_FILTER pruning
`plan_scan` caps the candidate set to the K most probable first tokens (the BAIT-Lite knob, now active).

In [ ]:
import numpy as np
from src.core.token_optimizer import plan_scan

probs = np.full(2000, 1e-7)
probs[[5, 11, 23, 42, 88, 101, 250, 900]] = [.30,.22,.15,.12,.08,.06,.04,.03]
for k in [None, 100, 20, 5]:
    plan = plan_scan(probs, prob_floor=1e-6, p=1.0, top_k=k)
    label = "no cap" if k is None else f"top_k={k}"
    print(f"{label:<8} -> {plan.n_candidates:>4} candidates scanned")

### 1.4 (Optional) Unit tests

In [ ]:
!pip install pytest -q
!python -m pytest tests/test_parallel_and_topk.py -v

---
# Path 2 — Real Ollama model  (the report's path)

Runs the scan against an actual local LLM served by **Ollama**. Use a **small** model on Colab
(`llama3.2:1b`) — large models will run out of RAM.


### 2.1 Install Ollama + start the server

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh
import subprocess, time
subprocess.Popen(["ollama", "serve"]); time.sleep(5)
print("Ollama server started.")

### 2.2 Pull a small model

In [ ]:
!ollama pull llama3.2:1b
!pip install ollama -q
print("Model ready.")

### 2.3 Run the black-box scan against the real model
Plain Ollama chat exposes no token distribution, so the Q-Score here is a transparent
text-agreement proxy (never the old fixed 0.8 placeholder).

In [ ]:
!python scripts/scan_blackbox.py --backend ollama --model llama3.2:1b \
    --top-k-filter 30 --parallel-workers 4 --max-target-length 8

---
# Path 3 — Full GPU model-zoo scan

Runs the **real research pipeline** (`scripts/scan.py`) on an actual backdoored 7B model from
the Hugging Face BAIT-ModelZoo, with the full enhanced suite **plus** the TOP_K_FILTER + parallel grafts.

**Before you start:**
- **Runtime:** `Runtime → Change runtime type →` **A100 GPU** + **High-RAM**. (L4 works; **not T4**.)
- **Disk:** the whole zoo is ~310 GB and won't fit. This downloads **one model at a time** (~15–30 GB).
- **HF token:** base models (Llama-2 / Mistral) are gated — request access on their HF pages + paste a token.
- **Time:** one 7B scan takes ~20–70 min.


### 3.1 Install the heavy GPU dependencies

In [ ]:
!pip install -q -r requirements.txt
!pip install -q loguru transformers accelerate peft datasets sentencepiece huggingface_hub
!pip install -e . -q
print("GPU deps installed.")

### 3.2 Confirm the GPU

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE — switch runtime to A100/L4")
!nvidia-smi --query-gpu=name,memory.total --format=csv

### 3.3 Hugging Face login (required — gated base models)
Token: https://huggingface.co/settings/tokens · accept the license on
[Llama-2-7b-hf](https://huggingface.co/meta-llama/Llama-2-7b-hf) and
[Mistral-7B-Instruct-v0.2](https://huggingface.co/mistralai/Mistral-7B-Instruct-v0.2).

In [ ]:
from huggingface_hub import login
HF_TOKEN = ""   # <-- paste your Hugging Face token
assert HF_TOKEN, "Paste your HF token (needed for the gated base model)."
login(HF_TOKEN); print("Logged in to Hugging Face.")

### 3.4 Download ONE model from the zoo
Pulls a single `id-XXXX` folder (adapter + config.json), not the whole 310 GB zoo.

In [ ]:
import os
MODEL_ID = "id-0001"     # change to scan a different model
os.makedirs("model_zoo/base_models", exist_ok=True)
!huggingface-cli download NoahShen/BAIT-ModelZoo \
    --include "models/{MODEL_ID}/*" --local-dir ./model_zoo_raw
!mkdir -p model_zoo/{MODEL_ID}
!cp -r model_zoo_raw/models/{MODEL_ID}/* model_zoo/{MODEL_ID}/
print("\nModel folder:"); !ls -la model_zoo/{MODEL_ID}/
print("\nconfig.json:"); !cat model_zoo/{MODEL_ID}/config.json

### 3.5 Run the full enhanced scan (with the grafts)
`--judge-backend none` avoids needing an OpenAI key. Set to `local` for an offline Llama-3 judge.

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
!python scripts/scan.py \
    --model-zoo-dir ./model_zoo --data-dir ./data \
    --cache-dir ./model_zoo/base_models --output-dir ./results \
    --run-name colab-path3 --model-id {MODEL_ID} \
    --judge-backend none --use-robust-qscore --prioritize-initial-tokens \
    --optimize-token-search --top-k-filter 500 --parallel-workers 4 \
    --use-baseline-calibration

### 3.6 View the result

In [ ]:
import json, glob
paths = glob.glob(f"results/colab-path3/{MODEL_ID}/result.json")
if paths: print(json.dumps(json.load(open(paths[0])), indent=2))
else: print("No result.json yet — check the scan output above.")

### 3.7 (Optional) scan all downloaded models + evaluate
Drop `--model-id` to scan every `id-*` in `model_zoo/`; add `--run-eval` for ROC-AUC / BLEU
(needs both benign and backdoored models scanned).

In [ ]:
!python scripts/scan.py \
    --model-zoo-dir ./model_zoo --data-dir ./data \
    --cache-dir ./model_zoo/base_models --output-dir ./results \
    --run-name colab-path3-all \
    --judge-backend none --use-robust-qscore --prioritize-initial-tokens \
    --optimize-token-search --top-k-filter 500 --parallel-workers 4 \
    --use-baseline-calibration --run-eval

---
# Graphs

Two independent sections:
- **Section A** — plots your team's committed reproduction numbers (90 models). No scanning needed.
- **Section B** — plots live Q-Scores from your Path-3 `result.json` files.


In [ ]:
import matplotlib.pyplot as plt, numpy as np, pandas as pd
plt.rcParams.update({"figure.dpi":120,"font.size":11,"axes.grid":True,"grid.alpha":0.3})
print("plotting ready")

## Graphs · Section A — Reproduction (your team's numbers)
From `reproduction_result/results.md`. Edit `rows` if your results change.

In [ ]:
rows = [
    ("alpaca",        20, "Mistral-7B", 1.000, 1.00, 1.000, 1.000, 1.000, 0.946, 1869.4),
    ("self-instruct", 10, "Mistral-7B", 1.000, 1.00, 1.000, 1.000, 1.000, 0.888, 4192.9),
    ("alpaca",        21, "Llama-2-7B", 0.952, 1.00, 0.900, 0.947, 0.950, 0.843, 1425.8),
    ("self-instruct", 10, "Llama-2-7B", 0.900, 1.00, 0.800, 0.889, 0.800, 0.740, 1659.6),
    ("alpaca",        19, "Llama-3-8B", 0.947, 1.00, 0.889, 0.941, 0.989, 0.844, 2894.5),
    ("self-instruct", 10, "Llama-3-8B", 1.000, 1.00, 1.000, 1.000, 1.000, 0.883, 4186.3),
]
df = pd.DataFrame(rows, columns=["dataset","n","model","acc","prec","rec","f1","auc","bleu","overhead"])
df

### A.1 Detection metrics by model type

In [ ]:
agg = df.groupby("model")[["acc","prec","rec","f1","auc"]].mean().reindex(["Mistral-7B","Llama-2-7B","Llama-3-8B"])
labels = ["Accuracy","Precision","Recall","F1","ROC-AUC"]
colors = ["#1f7a8c","#e09f3e","#3a8a5f","#8b5fb0","#c1444a"]
x = np.arange(len(agg)); w = 0.16
fig, ax = plt.subplots(figsize=(9,4.5))
for i,(m,l,c) in enumerate(zip(["acc","prec","rec","f1","auc"],labels,colors)):
    ax.bar(x + (i-2)*w, agg[m], w, label=l, color=c)
ax.set_xticks(x); ax.set_xticklabels(agg.index); ax.set_ylim(0.6,1.03); ax.set_ylabel("score")
ax.set_title("Detection metrics by base model (reproduction, 90 models)")
ax.legend(ncol=5, loc="lower center", bbox_to_anchor=(0.5,-0.28), frameon=False)
plt.tight_layout(); plt.savefig("fig_metrics_by_model.png", bbox_inches="tight"); plt.show()

### A.2 ROC-AUC and BLEU by dataset × model

In [ ]:
fig, axes = plt.subplots(1,2, figsize=(11,4.2))
for ax, col, title, ylim in [(axes[0],"auc","ROC-AUC (detection)",(0.7,1.02)),
                              (axes[1],"bleu","BLEU (inverted-target fidelity)",(0.6,1.0))]:
    piv = df.pivot(index="model", columns="dataset", values=col).reindex(["Mistral-7B","Llama-2-7B","Llama-3-8B"])
    piv.plot(kind="bar", ax=ax, color=["#1f7a8c","#e09f3e"], width=0.7)
    ax.set_title(title); ax.set_ylim(*ylim); ax.set_xlabel(""); ax.tick_params(axis="x", rotation=0)
    ax.legend(title="dataset", frameon=False)
plt.tight_layout(); plt.savefig("fig_auc_bleu.png", bbox_inches="tight"); plt.show()

### A.3 Confusion matrix (overall)
Note: results.md gives FP=0, FN=3 but not the exact benign/backdoored split; the counts below are
a consistent reconstruction (precision=1.0, recall=0.932). Correct them if you know the real split.

In [ ]:
TP, FN, FP, TN = 41, 3, 0, 46
cm = np.array([[TN, FP],[FN, TP]])
fig, ax = plt.subplots(figsize=(4.6,4.2)); ax.imshow(cm, cmap="Blues")
ax.set_xticks([0,1]); ax.set_yticks([0,1])
ax.set_xticklabels(["Pred benign","Pred backdoored"]); ax.set_yticklabels(["True benign","True backdoored"])
for i in range(2):
    for j in range(2):
        ax.text(j,i,cm[i,j],ha="center",va="center",
                color="white" if cm[i,j]>cm.max()/2 else "black", fontsize=15, weight="bold")
ax.set_title("Confusion matrix (overall)\nFP=0, FN=3")
plt.tight_layout(); plt.savefig("fig_confusion.png", bbox_inches="tight"); plt.show()

### A.4 Scan overhead by dataset × model

In [ ]:
piv = df.pivot(index="model", columns="dataset", values="overhead").reindex(["Mistral-7B","Llama-2-7B","Llama-3-8B"])
ax = piv.plot(kind="bar", figsize=(8,4), color=["#1f7a8c","#e09f3e"], width=0.7)
ax.set_ylabel("seconds / model"); ax.set_xlabel(""); ax.tick_params(axis="x", rotation=0)
ax.set_title("Average scan overhead per model"); ax.legend(title="dataset", frameon=False)
plt.tight_layout(); plt.savefig("fig_overhead.png", bbox_inches="tight"); plt.show()

### A.5 Overall summary (single slide figure)

In [ ]:
fig, ax = plt.subplots(figsize=(7,4))
labs = ["Accuracy","Precision","Recall","F1","ROC-AUC","BLEU"]; vals = [0.967,1.000,0.932,0.965,0.961,0.865]
bars = ax.bar(labs, vals, color="#1f2d50")
for b,v in zip(bars,vals): ax.text(b.get_x()+b.get_width()/2, v+0.008, f"{v:.3f}", ha="center", fontsize=10, weight="bold")
ax.set_ylim(0.8,1.03); ax.set_title("BAIT-Enhanced — overall reproduction (90 models)")
plt.tight_layout(); plt.savefig("fig_overall_summary.png", bbox_inches="tight"); plt.show()

## Graphs · Section B — Live-scan graphs (from your Path-3 runs)
Reads every `result.json` under `results/`. Run a Path-3 scan first, or point `RESULTS_DIR` at your saved results.

In [ ]:
import glob, json, os
RESULTS_DIR = "results"
records = []
for p in glob.glob(os.path.join(RESULTS_DIR, "**", "result.json"), recursive=True):
    try:
        d = json.load(open(p)); mid = os.path.basename(os.path.dirname(p))
        records.append({"model_id": mid,
                        "q_score": d.get("q_score", d.get("best_qscore", 0.0)),
                        "q_std": d.get("q_std", 0.0),
                        "invert_target": (d.get("invert_target") or d.get("best_sequence") or "")[:60]})
    except Exception as e: print("skip", p, e)
if not records:
    print("No result.json found under", RESULTS_DIR, "- run a Path-3 scan first.")
else:
    rdf = pd.DataFrame(records).sort_values("q_score", ascending=False)
    print(rdf.to_string(index=False))

### B.1 Q-Score per scanned model

In [ ]:
if records:
    rdf2 = rdf.reset_index(drop=True); TAU = 0.85
    colors = ["#c1444a" if q>TAU else "#3a8a5f" for q in rdf2["q_score"]]
    fig, ax = plt.subplots(figsize=(max(6,len(rdf2)*0.7),4))
    ax.bar(rdf2["model_id"], rdf2["q_score"], color=colors,
           yerr=rdf2["q_std"] if rdf2["q_std"].any() else None, capsize=4)
    ax.axhline(TAU, ls="--", color="#333", label=f"decision threshold {TAU}")
    ax.set_ylabel("Q-Score"); ax.set_ylim(0,1.05)
    ax.set_title("Q-Score per scanned model (red = flagged backdoored)")
    ax.legend(frameon=False); plt.xticks(rotation=45, ha="right")
    plt.tight_layout(); plt.savefig("fig_live_qscores.png", bbox_inches="tight"); plt.show()
else:
    print("No scans to plot yet.")

### Saved figures

In [ ]:
import glob
print("Figures saved (grab from the Files panel):")
for f in sorted(glob.glob("fig_*.png")): print(" ", f)

---
### Troubleshooting (Path 3)

| Symptom | Fix |
|---|---|
| `401 / gated repo` | Accept the license on the base model's HF page; use a token with access (3.3). |
| `CUDA out of memory` | You're on T4 — switch to **A100** (Runtime → Change runtime type). |
| `No space left on device` | Only download one `id-*` at a time (3.4), not the whole zoo. |
| `ModuleNotFoundError` | Re-run the install cells (Setup / 3.1). |

*BAIT-Enhanced · Group 7 · Paths 1–3 + graphs, single notebook.*